# LingBot Full-SFT FP32：6k → 累计10k

训练目标：现有HF 6k权重初始化，新增4000个优化器步骤；不使用LoRA。新优化器不是原8卡DCP精确续训。FP32模型/梯度/AdamW，CPU offload；单卡兼容补丁尚待GPU验证。逐节执行，禁止在输入与测试未通过时 Run all。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE=Path('/content/drive/MyDrive/LingBot-Colab-Stage2-20261008')
LOCAL=Path('/content/lingbot-stage2')
LOCAL.mkdir(parents=True,exist_ok=True)
import subprocess, json, shutil, os
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
print('Local disk free GiB:',shutil.disk_usage('/content').free/2**30)
gpu_info=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True).strip()
assert 'A100' in gpu_info and float(gpu_info.split(',')[-1])>=80000, '请选择A100 80GB运行时'
import psutil
assert psutil.virtual_memory().total/2**30>=140, 'Full-SFT CPU offload需要High-RAM环境'


## 输入缓存

full_sft_package.tar.gz 是本次FP32源码包；原 migration_package_v2 不再用于训练。先安装环境可不等待大权重；模型smoke仍必须完整校验权重与测试数据。正式4000步必须补齐50任务。


In [ ]:
import sys
import tarfile
archive=DRIVE/'full_sft_package.tar.gz'
assert archive.is_file()
with tarfile.open(archive) as f: f.extractall(LOCAL,filter='data')
source=DRIVE/'full_sft_6000'
subprocess.run([sys.executable,str(LOCAL/'package/assemble_weights.py'),str(source),str(LOCAL/'full_sft_6000')],check=True)
if (DRIVE/'lerobot').is_dir():
    shutil.copytree(DRIVE/'lerobot',LOCAL/'lerobot',dirs_exist_ok=True)
else:
    (LOCAL/'lerobot').mkdir(exist_ok=True)
    with tarfile.open(DRIVE/'smoke_adjust_bottle.tar.gz') as f: f.extractall(LOCAL/'lerobot',filter='data')


## CUDA 环境与FP32 CPU offload门槛

Python3.12 / Torch2.8.0独立环境。验证两次小批次梯度累积与普通FP32参考一致、AdamW在CPU更新、模型和优化器DCP保存恢复成功。这是基础机制测试，不能替代LingBot真实模型测试。


In [ ]:
subprocess.run(['bash',str(LOCAL/'package/setup_cuda.sh')],check=True)
PY='/content/lingbot-venv/bin/python'
subprocess.run([PY,'-m','torch.distributed.run','--standalone','--nproc-per-node=1',str(LOCAL/'package/test_full_sft_offload.py')],check=True)


## 补齐完整50任务数据（原版本重建）

若Drive中缺少完整数据，可使用同一固定HF版本和原XPolicyLab转换脚本，在独立LeRobot0.6.0环境下载并重建。此过程可能较长；检查2500episodes/549787frames后将结果保存到Drive。只需执行一次。


In [ ]:
subprocess.run([PY,str(LOCAL/'package/rebuild_data.py')],check=True)


## Teacher下载
首次smoke保留原结构与辅助损失，故需要teacher；不清空align_params。后续若移除辅助目标，须单独做结构保持补丁和权重加载对照。

In [ ]:
PY='/content/lingbot-venv/bin/python'
subprocess.run([PY,'-c',"from huggingface_hub import snapshot_download; snapshot_download('robbyant/lingbot-vla-v2-6b', revision='11c703bf6a5c1f45b3b69168482da11fdbba53d7',allow_patterns=['depth/model.pt','dino_video/teacher_step_10000.pth','dino_video/config.yaml'],local_dir='/content/lingbot-stage2/teachers/base'); snapshot_download('Ruicheng/moge-2-vitb-normal',revision='ca5f0e07ff01d3e5a364c1d954ed12ee1814b368',allow_patterns=['model.pt'],local_dir='/content/lingbot-stage2/teachers/moge')"],check=True)


## 两步真实Full-SFT FP32测试

保留原辅助目标。smoke global batch=2，测试梯度累积；正式global batch=256。要求所有模型参数可训练、浮点参数与梯度为FP32、有限loss/梯度及模型/优化器断点。全部Google Drive日志和断点写入专用新目录。


In [ ]:
subprocess.run([PY,str(LOCAL/'package/run_full_sft.py'),'--drive',str(DRIVE),'--mode','smoke'],check=True)


## 累计10k目标

从HF 6k权重初始化，新CPU AdamW优化器执行4000步。累计标签=6000+新增步骤；不是精确恢复原训练的优化器/调度器。正式保持global batch256（micro1、累积256），完整50任务2500episodes/549787frames通过校验后才允许启动。

新增强尚未物理验收，默认不加入。原8卡配置与单卡CPU offload吞吐差异必须实测；30小时不是完成承诺。FP32关闭BF16/TF32计算、FlashAttention BF16路径与MoE强制BF16内核；MoE保留权重布局，改用原生FP32线性层和autograd。

正式启动前必须恢复smoke断点到step3，确认global step和非空AdamW状态。正式训练日志每步落Drive，500步保存一次完整断点，最终4000步导出权重并记录累计10000。

RoboTwin assets与CUDA仿真环境未迁入，尚未运行闭环评测。


## 验证真实模型断点恢复到第三步


In [ ]:
subprocess.run([PY,str(LOCAL/'package/run_full_sft.py'),'--drive',str(DRIVE),'--mode','resume'],check=True)


## 正式新增4000步，到累计10000（需要完整50任务与恢复验证）


In [ ]:
subprocess.run([PY,str(LOCAL/'package/run_full_sft.py'),'--drive',str(DRIVE),'--mode','train'],check=True)
